# 08 — ระยะห่างจากทางหลวงหมายเลข 1 และ 2 และสัดส่วนประชากรริมถนนรายตำบล

**Part 1 → ตัวแปรการสัมผัสระดับตำบล** — ประชากรริมถนน (roadside population) ซึ่งเป็นกลุ่มเป้าหมายของวิทยานิพนธ์

มลพิษจากการจราจร (ไอเสียดีเซลรถบรรทุก ฝุ่นถนนฟุ้ง) **ลดลงเร็วมากตามระยะ** ส่วนใหญ่ภายใน 100–500 m จากขอบถนน (Karner et al., 2010; HEI, 2010)
ค่าดาวเทียม ~1 km (Notebook 06) จึงมองไม่เห็นความต่างนี้ notebook นี้วัด **จำนวนและสัดส่วนประชากรที่อาศัยใกล้ถนนพหลโยธิน (ทล.1) และมิตรภาพ (ทล.2)** ในแต่ละตำบล

| ขั้น | เนื้อหา |
|---|---|
| 1 | ข้อมูลจริง: ขอบเขตตำบล · ประชากร WorldPop 2020 100 m · เส้นถนน OpenStreetMap (Geofabrik) |
| 2 | เลือกเส้น ทล.1 / ทล.2 จากเลขทางหลวง (`ref`) และชื่อถนน + ตรวจด้วยแผนที่ |
| 3 | แผนที่ระยะห่างจากถนน (ความละเอียด 20 m) |
| 4 | กระจายประชากรแต่ละเซลล์ 100 m เป็น 5×5 จุดย่อย (ลดความหยาบของกริด) |
| 5 | ตัวชี้วัดรายตำบล: ระยะถ่วงประชากร · **สัดส่วนประชากรภายใน 500 m** · แถบ 0–150 / 150–300 / 300–500 m · **แยก ทล.1 / ทล.2 ที่ 150 / 300 / 500 m** · ดัชนีลดหลั่น ω(d) |
| 6 | ความไวต่อความกว้างแนวกันชน 50–1,000 m |
| 7 | ผลของความละเอียดกริด (จุดศูนย์กลางเซลล์ vs จุดย่อย) |
| 8 | แผนที่ บันทึกไฟล์ และข้อความในบทวิธีวิจัย |

> ปรับปรุง 25 ต.ค. 2569 ให้ตรงกับโครงร่างฉบับส่งบัณฑิตวิทยาลัย (หัวข้อ 4.4.10, 7.3.2, 7.3.5): เปลี่ยนแถบระยะจาก 0–115 / 115–570 m เป็น 0–150 / 150–300 / 300–500 m และเพิ่มสัดส่วนประชากรริมถนนแยก ทล.1 และ ทล.2
>
> `Runtime → Run all` ~5–10 นาที (ครั้งแรกดาวน์โหลดไฟล์ OSM ทั้งประเทศจาก Geofabrik ~1–3 นาที)

## 0. CONFIG

In [ ]:
PROVINCE_EN = "Saraburi"
BOUNDARY_SOURCE = "drive"       # "drive" (shapefile ตำบลใน Google Drive) → สำรอง "cod" → "gadm" → "osm" อัตโนมัติ
TAMBON_SHP = ""                 # path ไฟล์ขอบเขตตำบลใน Drive เช่น "GIS/tambon_saraburi.shp" (.shp/.gpkg/.zip/.geojson) — เว้นว่าง = ค้นหาอัตโนมัติ
TARGET_ROADS = {"HW1": {"refs": ["1"], "names": ["พหลโยธิน", "phahon"]},          # ถนนพหลโยธิน
                "HW2": {"refs": ["2"], "names": ["มิตรภาพ", "mittraphap", "mitraphap"]}}   # ถนนมิตรภาพ
MAJOR_CLASSES = ["motorway", "trunk", "primary", "motorway_link", "trunk_link", "primary_link"]
MAIN_BUFFER = 500                               # m — ตัวชี้วัดหลักตามโครงร่าง
BUFFERS = [50, 100, 150, 200, 300, 400, 500, 600, 750, 1000]
BANDS = [(0, 150), (150, 300), (300, 500)]      # แถบระยะภายในแนวกันชนหลัก 500 m (ขอบแถบ = 150/300/500 m ตามหัวข้อ 7.3.2)
ROAD_BUFFERS = [150, 300, 500]                  # m — สัดส่วนประชากรริมถนนแยก ทล.1 / ทล.2 (หัวข้อ 7.3.2, 7.3.5)
LAMBDAS = [100, 200, 300, 500]                  # m — ดัชนีลดหลั่น exp(−d/λ)
DIST_RES = 20                                   # m — ความละเอียดแผนที่ระยะห่าง
SUBPIX = 5                                      # จุดย่อยต่อด้านของเซลล์ WorldPop (5×5 = 25 จุด/เซลล์)
WORLDPOP_100M = True
WP_CACHE_DIR = "/content/drive/MyDrive/3-ColabProjects/Training/WorldPop_cache"   # ใช้ร่วมกับ Notebook 06
STATIONS_XY = {"24T": (100.871996, 14.685833), "25T": (100.926047, 14.526343)}
SAVE_TO_DRIVE = True

In [ ]:
import importlib.util, subprocess, sys
need = {"rasterio": "rasterio", "pyproj": "pyproj"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, io, sys, json, math, time, zipfile, warnings, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

UA = {"User-Agent": "Mozilla/5.0 (academic research notebook; PM2.5 Saraburi)"}

def setup_thai_font():
    """ให้กราฟ matplotlib แสดงภาษาไทยได้ (ฟอนต์เริ่มต้นไม่มีตัวอักษรไทย → แสดงเป็น □□□)"""
    import matplotlib.font_manager as fm, glob
    base = "/content" if os.path.isdir("/content") else "."
    path = os.path.join(base, "Sarabun-Regular.ttf")
    if not os.path.exists(path):
        try:
            r = requests.get("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", timeout=60)
            if r.status_code == 200 and len(r.content) > 50000: open(path, "wb").write(r.content)
        except Exception:
            pass
    if not os.path.exists(path):                                  # สำรอง: ฟอนต์ไทยจาก apt (TLWG)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1", shell=True)
        cand = sorted(glob.glob("/usr/share/fonts/**/tlwg/Loma.*tf", recursive=True)) + \
               sorted(glob.glob("/usr/share/fonts/**/Garuda.*tf", recursive=True))
        path = cand[0] if cand else None
    if path and os.path.exists(path):
        fm.fontManager.addfont(path)
        name = fm.FontProperties(fname=path).get_name()
        plt.rcParams["font.family"] = ["DejaVu Sans", name]      # ละติน = DejaVu, ไทย = fallback ไปฟอนต์ไทย
        print(f"✓ ฟอนต์ไทยสำหรับกราฟ: {name}")
    else:
        print("⚠ ติดตั้งฟอนต์ไทยไม่ได้ — ข้อความภาษาไทยในกราฟจะแสดงเป็นสี่เหลี่ยม")
setup_thai_font()

# เว็บราชการบางแห่ง (เช่น Air4Thai) ส่งใบรับรอง SSL ไม่ครบ chain → Python ตรวจสอบไม่ผ่าน (SSLError)
# อนุญาตให้ข้ามการตรวจใบรับรองเฉพาะโดเมนเหล่านี้ (ข้อมูลสาธารณะ อ่านอย่างเดียว)
SSL_RELAX_HOSTS = ("air4thai.pcd.go.th", "air4thai.com", "www.air4thai.com")

def http_get(url, timeout=(15, 120), tries=3, quiet=False, **kw):
    """GET พร้อม retry — คืน Response เมื่อ HTTP 200 ไม่เช่นนั้น raise"""
    last = None
    host = url.split("/")[2] if "://" in url else ""
    verify = kw.pop("verify", True)
    for k in range(tries):
        try:
            try:
                r = requests.get(url, headers=UA, timeout=timeout, verify=verify, **kw)
            except requests.exceptions.SSLError:
                if host not in SSL_RELAX_HOSTS or verify is False:
                    raise
                import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
                if not quiet: print(f"   (ใบรับรอง SSL ของ {host} ไม่สมบูรณ์ — เชื่อมต่อโดยไม่ตรวจใบรับรอง)")
                verify = False
                r = requests.get(url, headers=UA, timeout=timeout, verify=False, **kw)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
            if r.status_code == 429 and k < tries - 1:     # ขอถี่เกินโควตา (เช่น Open-Meteo) → รอนานขึ้นแล้วลองใหม่
                wait = int(r.headers.get("Retry-After", 0) or 0) or 30 * (k + 1)
                print(f"   HTTP 429 จาก {host} — รอ {wait} s แล้วลองใหม่", flush=True)
                time.sleep(wait); continue
        except Exception as e:
            last = type(e).__name__ + ": " + str(e)[:100]
        if k < tries - 1:
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"{last}  <- {url[:110]}")

def month_chunks(start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out, cur = [], s
    while cur <= e:
        nxt = min(cur + pd.offsets.MonthEnd(0), e)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt + pd.Timedelta(days=1)
    return out

from scipy import stats, ndimage
from pyproj import Transformer

In [ ]:
# ---- ที่เก็บผลลัพธ์ (ถ้า SAVE_TO_DRIVE=True จะเก็บใน Google Drive เพื่อให้ Notebook อื่นใช้ต่อได้) ----
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTDIR = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
    except Exception as e:
        print("⚠ mount Drive ไม่ได้ จะเก็บไว้ใน /content แทน:", e)
        OUTDIR = "/content/PM25_Saraburi_notebooks"
else:
    OUTDIR = "/content/PM25_Saraburi_notebooks" if os.path.isdir("/content") else "./PM25_Saraburi_notebooks"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR =", OUTDIR)

In [ ]:
# =====================================================================
#  เตรียมข้อมูลระดับตำบล (ดึงจากอินเทอร์เน็ตทั้งหมด)
#  1) ขอบเขตตำบล  : shapefile ใน Google Drive → OCHA COD-AB Thailand (มีรหัสตำบล 6 หลักตรงกับ HDC) → สำรอง GADM 4.1 → OpenStreetMap
#  2) ประชากร      : WorldPop 2020 (1 km, ปรับรวมรายตำบลด้วย zonal sum)
# =====================================================================
import geopandas as gpd
from shapely.geometry import Point
UTM = "EPSG:32647"

COD_BASE = "https://codgis.itos.uga.edu/arcgis/rest/services/COD_External/THA_pcode/FeatureServer"
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_THA_3.json.zip"

def load_tambons_cod():
    """OCHA COD-AB Thailand (ที่มา: กรมแผนที่ทหาร/RTSD) ผ่าน live service ของ ITOS — มีรหัสตำบล (pcode) และชื่อภาษาไทย"""
    import re
    from urllib.parse import quote
    meta = http_get(COD_BASE + "?f=json", timeout=(15, 60)).json()
    layers = meta.get("layers", [])
    lyr = next((l for l in layers if re.search(r"(adm(in)?\s*_?3|tambon|sub.?district)", l["name"], re.I)), None)
    if lyr is None:
        raise RuntimeError(f"ไม่พบชั้นข้อมูลระดับ 3: {[l['name'] for l in layers]}")
    for where in [f"ADM1_EN = '{PROVINCE_EN}'", "ADM1_PCODE = 'TH19'", f"ADM1_EN LIKE '%{PROVINCE_EN}%'"]:
        url = f"{COD_BASE}/{lyr['id']}/query?where={quote(where)}&outFields=*&outSR=4326&f=geojson"
        try:
            g = gpd.read_file(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content))
        except Exception:
            continue
        if len(g):
            break
    else:
        raise RuntimeError("query ไม่ได้ข้อมูล")
    g = g.set_crs("EPSG:4326", allow_override=True)
    namecol = next(c for c in ["ADM3_TH", "ADM3_EN", "admin3Name_th", "admin3Name_en"] if c in g)
    pc = next(c for c in ["ADM3_PCODE", "admin3Pcode"] if c in g)
    g["tambon_id"] = g[pc]; g["name"] = g[namecol]
    g["a_code"] = g[pc].astype(str).str.replace(r"\D", "", regex=True).str[:6]     # TH190101 → 190101 (ตรงกับ HDC)
    if "ADM3_EN" in g: g["name_en"] = g["ADM3_EN"]
    return g[["tambon_id", "a_code", "name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]], \
           f"OCHA COD-AB Thailand (ITOS live service, layer '{lyr['name']}')"

def load_tambons_gadm():
    """GADM 4.1 ระดับ 3 (ตำบล) — ไฟล์ทั้งประเทศ ~30–60 MB"""
    print("   ดาวน์โหลด GADM 4.1 THA level 3 ...")
    raw = http_get(GADM_URL, timeout=(15, 900)).content
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = next(n for n in z.namelist() if n.endswith(".json"))
        g = gpd.read_file(io.BytesIO(z.read(name)))
    g = g[g["NAME_1"].str.replace(" ", "").str.contains(PROVINCE_EN, case=False, na=False)].copy()
    g["tambon_id"] = g["GID_3"]
    g["name"] = g["NL_NAME_3"].where(g.get("NL_NAME_3", pd.Series(dtype=str)).notna() & (g.get("NL_NAME_3", "") != "NA"), g["NAME_3"]) \
        if "NL_NAME_3" in g else g["NAME_3"]
    g["name_en"] = g["NAME_3"]
    cols = ["tambon_id", "name", "name_en", "geometry"]
    if "CC_3" in g and g["CC_3"].astype(str).str.fullmatch(r"\d{6}").mean() > 0.8:
        g["a_code"] = g["CC_3"].astype(str); cols.insert(1, "a_code")
    return g[cols].to_crs("EPSG:4326"), "GADM 4.1 (THA level 3)"

def load_tambons_osm():
    import osmnx as ox
    ox.settings.timeout = 600
    prov = ox.geocode_to_gdf(f"{PROVINCE_EN} Province, Thailand")
    # หมายเหตุ: osmnx รวมแท็กแบบ OR → ถ้าใส่ทั้ง boundary และ admin_level จะได้ขอบเขตทุกระดับ (จังหวัด/อำเภอ/ตำบล) ปนกัน
    g = ox.features_from_polygon(prov.geometry.iloc[0], {"admin_level": "8"}).reset_index()
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if "boundary" in g: g = g[g["boundary"].astype(str) == "administrative"]
    if "admin_level" in g: g = g[g["admin_level"].astype(str) == "8"]
    if "element" in g and (g["element"] == "relation").any(): g = g[g["element"] == "relation"]
    g = g[g.representative_point().within(prov.geometry.iloc[0])]
    namecol = next(c for c in ["name:th", "name", "name:en"] if c in g)
    g["tambon_id"] = g.get("id", g.index).astype(str); g["name"] = g[namecol]
    if "name:en" in g: g["name_en"] = g["name:en"]
    return g[["tambon_id", "name", "geometry"] + (["name_en"] if "name_en" in g else [])], "OpenStreetMap admin_level=8"

# ---------- ขอบเขตตำบลจาก shapefile ใน Google Drive ----------
TAMBON_SEARCH_DIRS = ["/content/drive/MyDrive/PM25_Saraburi_notebooks/boundary",   # ที่แนะนำ
                      "/content/drive/MyDrive/PM25_Saraburi_notebooks",
                      "/content/drive/MyDrive"]                                    # ค้นหาลึกสุด 4 ชั้น
TAMBON_NAME_PAT = r"(tambon|sub.?district|subdist|adm3|admin3|ตำบล|tb_?|_tam)"
SARA_BBOX = (100.55, 14.25, 101.40, 15.05)
MISSING_PARTS = []
TAMBON_CACHE = "saraburi_tambon_boundary.gpkg"   # สำเนาเฉพาะสระบุรี (เล็ก อ่านเร็ว) สร้างอัตโนมัติข้างไฟล์ต้นฉบับ

def find_tambon_files():
    """คืนรายการ path ของ shapefile/gpkg/geojson/zip ที่อาจเป็นขอบเขตตำบลใน Drive (TAMBON_SHP ก่อน แล้วค้นหาอัตโนมัติ)"""
    import re, glob
    MISSING_PARTS.clear()
    if not os.path.isdir("/content/drive/MyDrive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"mount Drive ไม่ได้: {str(e)[:80]}")
    p = (globals().get("TAMBON_SHP") or "").strip()
    if p:
        if not p.startswith("/"): p = os.path.join("/content/drive/MyDrive", p)
        if os.path.exists(p): return [p]
        raise RuntimeError(f"ไม่พบไฟล์ TAMBON_SHP = {p}")
    exts = (".shp", ".gpkg", ".geojson", ".json", ".zip"); found = []
    for root0 in TAMBON_SEARCH_DIRS:
        if not os.path.isdir(root0): continue
        hits, base_depth = [], root0.rstrip("/").count("/")
        for root, dirs, files in os.walk(root0):
            if root.count("/") - base_depth >= 4: dirs[:] = []
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("SatPM25_cache", "ERA5_cache", "WorldPop_cache", "GDAS", "PCD_PM25")]
            for fn in files:
                if fn.lower().endswith(".shx") and re.search(TAMBON_NAME_PAT, fn, re.I) and \
                        (fn[:-4] + ".shp").lower() not in {f.lower() for f in files}:
                    MISSING_PARTS.append(os.path.join(root, fn[:-4] + ".shp"))         # มี .shx แต่ไม่มี .shp
                anyname = root.rstrip("/").endswith("/boundary")          # ในโฟลเดอร์ boundary รับทุกชื่อไฟล์
                if fn.lower().endswith(exts) and (anyname or re.search(TAMBON_NAME_PAT, fn, re.I)) \
                        and not re.search(r"(rates|gistar|lisa|roadside|summary|result)", fn, re.I):
                    if fn.lower().endswith(".shp"):
                        stem = fn[:-4]; have = {f.lower() for f in files}
                        if not {stem.lower() + ".dbf", stem.lower() + ".shx"} <= have:
                            MISSING_PARTS.append(os.path.join(root, fn)); continue      # .dbf/.shx ยังไม่ครบ (อัปโหลดไม่เสร็จ?)
                    hits.append(os.path.join(root, fn))
            if root0 == "/content/drive/MyDrive" and len(hits) > 50: break
        if hits:
            order = {".shp": 0, ".gpkg": 1, ".zip": 2, ".geojson": 3, ".json": 4}
            rank = lambda h: 0 if re.search(TAMBON_NAME_PAT, os.path.basename(h), re.I) else \
                             2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|_ln|_pt|line|point)", os.path.basename(h), re.I) else 1
            hits.sort(key=lambda h: (rank(h), order[os.path.splitext(h)[1].lower()], len(h)))
            found += [h for h in hits if h not in found]
            if any(os.path.basename(h) == TAMBON_CACHE for h in found): break      # เจอสำเนาแล้ว ไม่ต้องค้นต่อ
    found.sort(key=lambda h: os.path.basename(h) != TAMBON_CACHE)                # สำเนาเฉพาะสระบุรีมาก่อน
    if found: return found
    extra = f" (พบ .shp ที่ไฟล์ประกอบไม่ครบ: {[os.path.basename(m) for m in MISSING_PARTS[:3]]})" if MISSING_PARTS else ""
    raise RuntimeError("ไม่พบ shapefile ตำบลใน Drive" + extra + " — วางไฟล์ไว้ที่ MyDrive/PM25_Saraburi_notebooks/boundary/ "
                       "(ชื่อไฟล์มีคำว่า tambon/subdistrict/ตำบล) หรือกำหนด TAMBON_SHP")

def read_tambon_file(path):
    """อ่านไฟล์ขอบเขต + แก้ภาษาไทยเพี้ยน (TIS-620) + กำหนด CRS + ตัดเฉพาะสระบุรี + หาคอลัมน์รหัส/ชื่อตำบล"""
    import re
    src, layer = path, None
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            inner = [n for n in z.namelist() if n.lower().endswith((".shp", ".gpkg", ".geojson", ".json"))]
        if not inner: raise RuntimeError(f"ใน {os.path.basename(path)} ไม่มี .shp/.gpkg/.geojson")
        lvl = lambda n: (0 if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I) else
                         2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|line|pt|point)", n, re.I) else 1)
        inner.sort(key=lambda n: (lvl(n), len(n)))
        src = f"zip://{path}!{inner[0]}"
    if src.lower().endswith(".gpkg"):
        try:
            import pyogrio
            names = [l[0] for l in pyogrio.list_layers(src)]
        except Exception:
            names = []
        if len(names) > 1:
            pref = [n for n in names if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I)]
            layer = pref[0] if pref else names[-1]
    kw = {"layer": layer} if layer else {}
    g = None
    if os.path.getsize(path) > 50e6:                    # ไฟล์ทั้งประเทศขนาดใหญ่ → อ่านเฉพาะกรอบสระบุรี (เร็วกว่ามาก)
        try:
            g = gpd.read_file(src, bbox=(SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3), **kw)
            if len(g) < MIN_TAMBONS: g = None           # พิกัดในไฟล์ไม่ใช่ lat/lon → อ่านทั้งไฟล์
            else: kw["bbox"] = (SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3)
        except Exception:
            g = None
    if g is None: g = gpd.read_file(src, **kw)
    thai = lambda s: bool(re.search(r"[฀-๿]", str(s)))
    garbled = lambda s: bool(re.search(r"[À-ÿ]{2,}", str(s)))
    is_txt = lambda c: c != "geometry" and not pd.api.types.is_numeric_dtype(g[c])
    obj = [c for c in g.columns if is_txt(c)]
    if obj and not any(g[c].astype(str).map(thai).any() for c in obj) and any(g[c].astype(str).map(garbled).any() for c in obj):
        for enc in ("cp874", "tis-620"):
            try:
                g2 = gpd.read_file(src, encoding=enc, **kw)
                if any(g2[c].astype(str).map(thai).any() for c in obj if c in g2): g = g2; obj = [c for c in g.columns if is_txt(c)]; print(f"   อ่านภาษาไทยด้วย encoding={enc}"); break
            except Exception: pass
    g = g[g.geometry.notna() & g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if g.empty: raise RuntimeError("ไม่มี polygon (อาจเป็นไฟล์จุด/เส้น)")
    if g.crs is None:
        x0, y0, x1, y1 = g.total_bounds
        if -180 <= x0 <= 180 and -90 <= y0 <= 90: g = g.set_crs("EPSG:4326")
        else: g = g.set_crs(UTM); print("   ⚠ ไฟล์ไม่มี .prj — สมมติเป็น UTM 47N (EPSG:32647)")
    g = g.to_crs("EPSG:4326")
    up = {c: c.upper() for c in g.columns}
    digits = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    # ---- รหัสตำบล 6 หลัก ----
    code = None
    for c in g.columns:
        if c == "geometry": continue
        d = digits(g[c])
        if (d.str.len() == 6).mean() > 0.8 and d.str[:2].nunique() <= 80: code = d.where(d.str.len() == 6); break
    if code is None:
        pc = next((c for c in g.columns if re.fullmatch(r"(P|PV|PROV|PRO|CHANGWAT|ADM1)_?(CODE|ID|CD)", up[c])), None)
        ac = next((c for c in g.columns if re.fullmatch(r"(A|AP|AMP|AMPHOE|DIST|ADM2)_?(CODE|ID|CD)", up[c])), None)
        tc = next((c for c in g.columns if re.fullmatch(r"(T|TB|TAM|TAMBON|SUBDIST|ADM3)_?(CODE|ID|CD)", up[c])), None)
        if pc and ac and tc:
            p, a, t = digits(g[pc]).str[-2:].str.zfill(2), digits(g[ac]).str[-2:].str.zfill(2), digits(g[tc]).str[-2:].str.zfill(2)
            code = p + a + t
    # ---- ตัดเฉพาะจังหวัดสระบุรี ----
    n_all = len(g); keep = None
    if code is not None and (code.str[:2] == "19").sum() >= MIN_TAMBONS: keep = code.str[:2] == "19"
    if keep is None:
        for c in g.columns:
            if is_txt(c):
                k = g[c].astype(str).str.contains(r"สระบุรี|Saraburi", case=False, na=False)
                if MIN_TAMBONS <= k.sum() < len(g): keep = k; break
    if keep is None and n_all > 140:
        rp = g.representative_point(); w, s, e, n = SARA_BBOX
        keep = rp.x.between(w, e) & rp.y.between(s, n)
        print("   ⚠ ไม่พบคอลัมน์จังหวัด — ตัดด้วยกรอบพิกัดโดยประมาณ (ตรวจตำบลริมขอบจังหวัดอีกครั้ง)")
    if keep is not None:
        g = g[keep.values].copy(); code = code[keep.values] if code is not None else None
    # ---- ชื่อตำบล ----
    pick = lambda pats: next((c for p in pats for c in g.columns if re.fullmatch(p, up[c])), None)
    nc = pick([r"TAM_?NAMT", r"T_?NAME_?T", r"TB_?TN", r"TAMBON_?T(H)?", r"ADM3_?TH", r"NAME_?TH", r"TAM_?NAME", r"T_?NAME", r"TAMBON", r"ADM3_?EN", r"NAME_?3", r"NAME"])
    if nc is None:
        nc = next((c for c in g.columns if is_txt(c) and g[c].astype(str).map(thai).mean() > 0.8), None)
    ec = pick([r"TAM_?NAME_?E", r"TAM_?NAMEE", r"T_?NAME_?E", r"TB_?EN", r"ADM3_?EN", r"NAME_?EN", r"NAME_?3"])
    g["name"] = g[nc].astype(str).str.replace(r"^(ตำบล|ต\.)\s*", "", regex=True) if nc else [f"T{i+1:03d}" for i in range(len(g))]
    if ec and ec != nc: g["name_en"] = g[ec].astype(str)
    if code is not None: g["a_code"] = code.values
    g["tambon_id"] = g["a_code"] if "a_code" in g else g["name"]
    cols = ["tambon_id"] + (["a_code"] if "a_code" in g else []) + ["name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]
    info = f"รหัส={'มี' if 'a_code' in g else 'ไม่มี'}, ชื่อ={nc}, ตัดจาก {n_all} → {len(g)} หน่วย"
    return g[cols].reset_index(drop=True), info

def load_tambons_drive():
    import time
    errs = []; t0 = time.time()
    for path in find_tambon_files()[:6]:
        short = path.replace("/content/drive/MyDrive/", "MyDrive/")
        try:
            g, info = read_tambon_file(path)
            if len(g) < MIN_TAMBONS: raise RuntimeError(f"ได้ {len(g)} หน่วย ({info})")
            print(f"   ไฟล์: {short} ({info}; {time.time() - t0:.0f} s)")
            if os.path.basename(path) != TAMBON_CACHE:
                try:                                      # ครั้งต่อไปอ่านสำเนานี้แทน (ไม่กี่วินาที)
                    cp = os.path.join(os.path.dirname(path), TAMBON_CACHE)
                    g.to_file(cp, driver="GPKG"); print(f"   บันทึกสำเนาเฉพาะสระบุรี → {cp.replace('/content/drive/MyDrive/', 'MyDrive/')}")
                except Exception as e:
                    print(f"   (บันทึกสำเนาไม่ได้: {str(e)[:60]})")
            return g, f"Drive: {os.path.basename(path)}"
        except Exception as e:
            errs.append(f"{os.path.basename(path)}: {str(e)[:70]}")
    raise RuntimeError("ไฟล์ใน Drive ใช้ไม่ได้ — " + " | ".join(errs))

MIN_TAMBONS = 60     # สระบุรีมี ~108–111 ตำบล ถ้าได้น้อยกว่านี้แสดงว่าได้ขอบเขตผิดระดับ (เช่น อำเภอ)

def clean_units(g):
    """ตัด polygon ซ้ำ และ polygon 'แม่' ที่ครอบหน่วยอื่น ≥ 2 หน่วย (เช่น อำเภอ/จังหวัดที่ปนมา)"""
    n0 = len(g)
    g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["geometry"] = g.geometry.buffer(0)
    key = g.to_crs(UTM).geometry.apply(lambda x: (round(x.area / 1e4), round(x.centroid.x / 50), round(x.centroid.y / 50)))
    g = g[~key.duplicated()].reset_index(drop=True)
    rp = g.representative_point(); sidx = rp.sindex
    area = g.to_crs(UTM).area.values
    parent = []
    for i, geom in enumerate(g.geometry):
        inside = [j for j in sidx.query(geom, predicate="contains") if j != i and area[j] < 0.8 * area[i]]
        parent.append(len(inside) >= 2)               # ครอบหน่วยที่เล็กกว่าอย่างน้อย 2 หน่วย = ขอบเขตระดับบน
    g = g[~np.array(parent)].reset_index(drop=True)
    if "name" in g:
        g = g[~g["name"].astype(str).str.match(r"^(อำเภอ|จังหวัด|เขต)")].reset_index(drop=True)
    if len(g) != n0:
        print(f"   ทำความสะอาดขอบเขต: {n0} → {len(g)} หน่วย (ตัด polygon ซ้ำ/ระดับอำเภอ-จังหวัดที่ปนมา)")
    return g

KV_URL = "https://raw.githubusercontent.com/kongvut/thai-province-data/master/api/latest/sub_district.json"

def add_tambon_codes(g):
    """เติมรหัสตำบล 6 หลัก (มท.) ให้ขอบเขตที่ไม่มีรหัส (GADM/OSM): จับคู่ตำแหน่งตำบลก่อน แล้วจับคู่ชื่อ"""
    import re
    js = http_get(KV_URL, timeout=(15, 120)).json()
    kv = pd.DataFrame([{"a_code": str(t["id"]), "th": (t["name"] or {}).get("th", ""), "en": (t["name"] or {}).get("en", ""),
                        "lat": t.get("lat"), "lon": t.get("long")} for t in js if t.get("id")])
    pts = kv.dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs="EPSG:4326")
    j = gpd.sjoin(pts, g[["geometry"]].reset_index().rename(columns={"index": "gi"}), predicate="within")
    prov = j.a_code.str[:2].mode().iloc[0]
    kv = kv[kv.a_code.str[:2] == prov]
    norm_th = lambda x: re.sub(r"^(ตำบล|ต\.)", "", str(x)).replace(" ", "").strip()
    norm_en = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
    by_th = dict(zip(kv.th.map(norm_th), kv.a_code)); by_en = dict(zip(kv.en.map(norm_en), kv.a_code))
    code = pd.Series(index=g.index, dtype=object)
    for col, table, f in [("name", by_th, norm_th), ("name_en", by_en, norm_en), ("name", by_en, norm_en)]:
        if col in g:
            m = g[col].map(f).map(table); code = code.fillna(m)
    jj = j[j.a_code.str[:2] == prov].drop_duplicates("gi").set_index("gi")["a_code"]
    code = code.fillna(jj.reindex(g.index))
    g["a_code"] = code
    th = dict(zip(kv.a_code, kv.th)); g["name_th"] = g["a_code"].map(th)
    if g["name_th"].notna().mean() > 0.8: g["name"] = g["name_th"].fillna(g["name"])
    print(f"   เติมรหัสตำบล (มท.) ได้ {g.a_code.notna().sum()} / {len(g)} หน่วย (จังหวัดรหัส {prov})")
    return g.drop(columns="name_th")

def load_tambons():
    loaders = {"drive": load_tambons_drive, "cod": load_tambons_cod, "gadm": load_tambons_gadm, "osm": load_tambons_osm}
    first = BOUNDARY_SOURCE if BOUNDARY_SOURCE in loaders else "drive"
    order = [loaders[first]] + [f for k, f in loaders.items() if k != first]
    for f in order:
        try:
            g, src = f(); g = clean_units(g.reset_index(drop=True))
            if len(g) < MIN_TAMBONS:
                raise RuntimeError(f"ได้เพียง {len(g)} หน่วย — น่าจะไม่ใช่ระดับตำบล")
            if "a_code" not in g or g["a_code"].isna().mean() > 0.2:
                try:
                    g = add_tambon_codes(g)
                except Exception as e:
                    print(f"   ⚠ เติมรหัสตำบลไม่ได้: {str(e)[:80]} — การจับคู่ข้อมูล HDC จะใช้ชื่อแทน")
            print(f"✓ ขอบเขตตำบล: {len(g)} ตำบล  ({src})"); return g, src
        except Exception as e:
            print(f"   ✗ {f.__name__}: {str(e)[:120]}")
    raise RuntimeError("โหลดขอบเขตตำบลไม่สำเร็จ")

WORLDPOP_URLS = [
    "https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/2020/THA/tha_ppp_2020_1km_Aggregated_UNadj.tif",  # ~3 MB
    "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/THA/tha_ppp_2020_UNadj.tif",                            # ~250 MB (100 m)
]

def add_population(g):
    """ดาวน์โหลด GeoTIFF ของ WorldPop ลงเครื่อง แล้วรวมประชากรรายตำบล (zonal sum)"""
    import rasterio
    from rasterio.mask import mask
    for u in [WORLDPOP_URLS[0]] + WORLDPOP_URLS:          # ไฟล์ 1 km ลอง 2 ครั้ง (server ตัดการเชื่อมต่อบ่อย) แล้วจึงใช้ 100 m
        fn = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
        try:
            if not (os.path.exists(fn) and os.path.getsize(fn) > 1e6):
                print(f"   ดาวน์โหลด {u.split('/')[-1]} ...")
                with requests.get(u, headers=UA, stream=True, timeout=(15, 600)) as r:
                    r.raise_for_status()
                    with open(fn, "wb") as f:
                        for chunk in r.iter_content(1 << 20): f.write(chunk)
            with rasterio.open(fn) as src:
                gg = g.to_crs(src.crs); pop = []
                for geom in gg.geometry:
                    arr, _ = mask(src, [geom], crop=True, all_touched=False, filled=True, nodata=0)
                    a = arr[0].astype(float); a[~np.isfinite(a) | (a < 0)] = 0; pop.append(a.sum())
            if sum(pop) <= 0:
                raise RuntimeError("ผลรวมประชากร = 0")
            g["pop"] = np.round(pop); print(f"✓ ประชากร WorldPop 2020: รวม {g['pop'].sum():,.0f} คน  ({u.split('/')[-1]})")
            return g, "WorldPop 2020 UN-adjusted (" + u.split("/")[-1] + ")"
        except Exception as e:
            print(f"   ✗ WorldPop {u.split('/')[-1]}: {type(e).__name__}: {str(e)[:100]}")
            if os.path.exists(fn) and os.path.getsize(fn) < 1e6: os.remove(fn)
    print("⚠ ไม่ได้ข้อมูลประชากร — ใช้พื้นที่แทนเป็นตัวหารสำรอง"); g["pop"] = np.nan
    return g, None

FIRMS_PRODUCTS = ["viirs-snpp", "viirs-noaa20", "modis"]       # ลำดับความชอบ (VIIRS 375 m ละเอียดกว่า MODIS 1 km)
FIRMS_API_SRC = {"viirs-snpp": ["VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"], "viirs-noaa20": ["VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"],
                 "modis": ["MODIS_SP", "MODIS_NRT"]}


# ---- WorldPop: อ่านเฉพาะกรอบพื้นที่จากไฟล์ 100 m ผ่าน HTTP (ไม่ต้องดาวน์โหลดทั้งประเทศ) → สำรอง: ไฟล์ 1 km ----
def _wp_cache_path(bounds, u):
    import hashlib
    d = globals().get("WP_CACHE_DIR") or ""
    if d.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        d = os.path.join(OUTDIR, "WorldPop_cache")                     # Drive ไม่ได้ mount → เก็บใน OUTDIR
    d = d or os.path.join(OUTDIR, "WorldPop_cache"); os.makedirs(d, exist_ok=True)
    tag = hashlib.md5(str(np.round(bounds, 4)).encode()).hexdigest()[:8]
    return os.path.join(d, u.split("/")[-1].replace(".tif", f"_{tag}.tif"))

def worldpop_window(bounds, prefer_100m=True):
    """คืน (array ประชากรต่อเซลล์, affine transform, crs, คำอธิบายแหล่ง) เฉพาะกรอบ bounds = (W, S, E, N) องศา
    ลำดับ: ① ไฟล์ที่ตัดกรอบไว้แล้วใน Drive (WP_CACHE_DIR) → ② อ่านบางส่วนผ่าน HTTP → ③ ดาวน์โหลดทั้งไฟล์
    ได้ผลแล้วจะเก็บเฉพาะกรอบลง Drive (ไม่กี่ MB) — ครั้งต่อไปไม่ต้องดาวน์โหลด 263 MB อีก (ใช้ร่วมกันระหว่าง Notebook 06 และ 08)"""
    import rasterio
    from rasterio.windows import from_bounds
    urls = (WORLDPOP_URLS[1:] if prefer_100m else []) + WORLDPOP_URLS[:1]
    for u in urls:
        cp = _wp_cache_path(bounds, u)
        if os.path.exists(cp):
            with rasterio.open(cp) as src:
                arr = src.read(1).astype("float64"); tr = src.transform; crs = src.crs
            label = f"WorldPop 2020 UN-adjusted ({u.split('/')[-1]}, cache: {cp})"
            print(f"✓ {label}: {arr.shape[1]}×{arr.shape[0]} เซลล์, รวม {arr.sum():,.0f} คน")
            return arr, tr, crs, label
    for u in urls:
        for how in ("vsicurl", "download"):
            try:
                if how == "vsicurl":
                    path = "/vsicurl/" + u
                    env = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="3",
                               CPL_VSIL_CURL_CHUNK_SIZE=str(1 << 20))
                else:
                    path = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
                    if not (os.path.exists(path) and os.path.getsize(path) > 1e6):
                        print(f"   ดาวน์โหลด {u.split('/')[-1]} ทั้งไฟล์ ...", flush=True)
                        with requests.get(u, headers=UA, stream=True, timeout=(15, 900)) as r:
                            r.raise_for_status()
                            with open(path + ".part", "wb") as f:
                                for ch in r.iter_content(1 << 22): f.write(ch)
                        os.replace(path + ".part", path)
                    env = {}
                with rasterio.Env(**env), rasterio.open(path) as src:
                    win = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
                    arr = src.read(1, window=win, boundless=True, fill_value=0).astype("float64")
                    tr = src.window_transform(win); crs = src.crs
                arr[~np.isfinite(arr) | (arr < 0)] = 0
                if arr.sum() <= 0: raise RuntimeError("ผลรวมประชากร = 0")
                label = f"WorldPop 2020 UN-adjusted ({u.split('/')[-1]}, {how})"
                print(f"✓ {label}: {arr.shape[1]}×{arr.shape[0]} เซลล์, รวม {arr.sum():,.0f} คน")
                try:                                                   # เก็บเฉพาะกรอบไว้ใช้ครั้งต่อไป
                    cp = _wp_cache_path(bounds, u)
                    with rasterio.open(cp, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                                       dtype="float32", crs=crs, transform=tr, compress="deflate") as dst:
                        dst.write(arr.astype("float32"), 1)
                    print(f"   เก็บเฉพาะกรอบไว้ที่ {cp} ({os.path.getsize(cp) / 1e6:.1f} MB) — ครั้งต่อไปไม่ต้องดาวน์โหลด")
                    if how == "download" and os.path.getsize(path) > 1e8:
                        os.remove(path)                                # ลบไฟล์ทั้งประเทศ (263 MB) ออกจากเครื่อง Colab
                except Exception as e:
                    print(f"   ⚠ เก็บ cache ไม่ได้: {str(e)[:80]}")
                return arr, tr, crs, label
            except Exception as e:
                print(f"   ✗ WorldPop {u.split('/')[-1]} ({how}): {type(e).__name__}: {str(e)[:90]}")
    raise RuntimeError("อ่าน WorldPop ไม่ได้ — ลองรันเซลล์ใหม่ภายหลัง")

# ---- เรียก Overpass API โดยตรง (timeout สั้น, ลองหลาย server, แสดงความคืบหน้า) ----
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import polygonize, unary_union
OVERPASS = ["https://overpass-api.de/api/interpreter", "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
            "https://overpass.private.coffee/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]
# overpass-api.de ตอบ HTTP 406 กับ User-Agent บางแบบ → ใช้ UA ที่ระบุตัวตนชัดเจนตามนโยบายการใช้งาน
OP_HEADERS = {"User-Agent": "PM25-Saraburi-research-notebook/1.1 (academic; Google Colab)", "Accept": "*/*"}

GEOFABRIK_URL = "https://download.geofabrik.de/asia/thailand-latest-free.shp.zip"
GF_LAYERS = {"roads": "gis_osm_roads_free_1.shp", "landuse_a": "gis_osm_landuse_a_free_1.shp",
             "pois_a": "gis_osm_pois_a_free_1.shp", "pois": "gis_osm_pois_free_1.shp"}

def _area_bbox(area):
    a = area.strip("()")
    if a.startswith("around:"):
        r_, lat_, lon_ = map(float, a.split(":")[1].split(","))
        dlat = r_ / 111320; dlon = dlat / np.cos(np.deg2rad(lat_))
        return (lon_ - dlon, lat_ - dlat, lon_ + dlon, lat_ + dlat), (lat_, lon_, r_)
    s_, w_, n_, e_ = map(float, a.split(",")); return (w_, s_, e_, n_), None

def geofabrik_gdf(gf, area):
    """ไฟล์ OSM ทั้งประเทศจาก Geofabrik (shapefile อัปเดตทุกวัน) — ดาวน์โหลดครั้งเดียว ~0.5–1 GB แล้วอ่านเฉพาะกรอบพื้นที่"""
    layer, fclasses, col = gf
    fn = ("/content" if os.path.isdir("/content") else ".") + "/thailand-latest-free.shp.zip"
    t0 = time.time()
    try:
        if not (os.path.exists(fn) and os.path.getsize(fn) > 1e8):
            print("   ดาวน์โหลด Geofabrik thailand-latest-free.shp.zip (ครั้งเดียว, ~1–3 นาที) ...", flush=True)
            with requests.get(GEOFABRIK_URL, headers=OP_HEADERS, stream=True, timeout=(15, 600)) as r:
                r.raise_for_status()
                with open(fn + ".part", "wb") as f:
                    for ch in r.iter_content(1 << 22): f.write(ch)
            os.replace(fn + ".part", fn)
        bbox, circ = _area_bbox(area)
        g = gpd.read_file(f"/vsizip/{fn}/{GF_LAYERS[layer]}", bbox=bbox)
        if fclasses: g = g[g["fclass"].isin(fclasses)]
        if circ:
            lat_, lon_, r_ = circ
            d = g.to_crs(UTM).distance(gpd.GeoSeries([Point(lon_, lat_)], crs="EPSG:4326").to_crs(UTM).iloc[0])
            g = g[d <= r_]
        g = g.rename(columns={"fclass": col}).copy()
        print(f"   ✓ Geofabrik {layer}: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g.to_crs("EPSG:4326")
    except Exception as e:
        print(f"   ✗ Geofabrik ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:90]}", flush=True)
        if os.path.exists(fn + ".part"): os.remove(fn + ".part")
        return None

def ohsome_gdf(ohsome_filter, area, kind):
    """ohsome API (HeiGIT) — ดึง OSM ตามตัวกรอง; เสถียรกว่า Overpass ที่มักเต็ม (502 at capacity)"""
    a = area.strip("()")
    if a.startswith("around:"):
        r_, lat_, lon_ = a.split(":")[1].split(","); geo = {"bcircles": f"{lon_},{lat_},{r_}"}
    else:
        s_, w_, n_, e_ = a.split(","); geo = {"bboxes": f"{w_},{s_},{e_},{n_}"}
    gfilt = "geometry:line" if kind == "line" else "(geometry:polygon or geometry:point)"
    t0 = time.time()
    try:
        r = requests.post("https://api.ohsome.org/v1/elements/geometry",
                          data={**geo, "filter": f"({ohsome_filter}) and {gfilt}", "properties": "tags"},
                          headers=OP_HEADERS, timeout=(15, 240))
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:80]}")
        feats = r.json().get("features", [])
        g = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326") if feats else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        print(f"   ✓ ohsome: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g
    except Exception as e:
        print(f"   ✗ ohsome ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:80]}", flush=True)
        return None

def overpass_gdf(selectors, area, kind, timeout_s=90, ohsome_filter=None, cache_key=None, gf=None):
    """ลำดับ: cache ในเครื่อง → Geofabrik (ไฟล์) → ohsome → Overpass หลาย server"""
    import hashlib
    cdir = "/content/osm_cache" if os.path.isdir("/content") else "osm_cache"; os.makedirs(cdir, exist_ok=True)
    cfile = os.path.join(cdir, hashlib.md5((str(selectors) + area + kind).encode()).hexdigest() + ".gpkg")
    if os.path.exists(cfile):
        g = gpd.read_file(cfile); print(f"   ✓ ใช้ข้อมูล OSM ที่เก็บไว้ ({len(g):,} รายการ)"); return g
    g = geofabrik_gdf(gf, area) if gf else None
    if g is None or g.empty:
        g = ohsome_gdf(ohsome_filter, area, kind) if ohsome_filter else None
    if g is None or g.empty:
        g = _overpass_raw(selectors, area, kind, timeout_s)
    if len(g):
        try: g.to_file(cfile, driver="GPKG")
        except Exception: pass
    return g

def _overpass_raw(selectors, area, kind, timeout_s=90):
    # selectors เช่น ['way["landuse"="quarry"]'] ; area = '(s,w,n,e)' หรือ '(around:15000,lat,lon)'
    # kind = 'polygon' (way ปิด + relation + node) หรือ 'line'
    q = f"[out:json][timeout:{timeout_s}];(" + "".join(f"{sel}{area};" for sel in selectors) + ");out geom tags;"
    for url in [u for u in OVERPASS for _ in range(2)]:
        t0 = time.time()
        try:
            r = requests.post(url, data={"data": q}, headers=OP_HEADERS, timeout=(15, timeout_s + 15))
            if r.status_code in (403, 405, 406):                     # บาง server ไม่รับ POST/UA นี้ → ลอง GET
                r = requests.get(url, params={"data": q}, headers=OP_HEADERS, timeout=(15, timeout_s + 15))
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:60].strip()}")
            els = r.json().get("elements", [])
        except Exception as e:
            print(f"   ✗ {url.split('/')[2]} ({time.time()-t0:.0f} s): {type(e).__name__}: {str(e)[:70]}", flush=True)
            time.sleep(10); continue
        rows = []
        for el in els:
            geom = None
            if el["type"] == "way" and "geometry" in el:
                xy = [(p["lon"], p["lat"]) for p in el["geometry"]]
                if kind == "line" and len(xy) >= 2: geom = LineString(xy)
                if kind == "polygon" and len(xy) >= 4 and xy[0] == xy[-1]: geom = Polygon(xy)
            elif el["type"] == "relation" and kind == "polygon":
                lines = [LineString([(p["lon"], p["lat"]) for p in m["geometry"]]) for m in el.get("members", [])
                         if m.get("role", "outer") in ("outer", "") and len(m.get("geometry", [])) >= 2]
                polys = list(polygonize(unary_union(lines))) if lines else []
                if polys: geom = unary_union(polys)
            elif el["type"] == "node" and kind == "polygon" and "lat" in el:
                geom = Point(el["lon"], el["lat"])
            if geom is not None and not geom.is_empty:
                rows.append({**el.get("tags", {}), "osm_type": el["type"], "osm_id": el["id"], "geometry": geom})
        g = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326") if rows else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
        print(f"   ✓ {url.split('/')[2]}: {len(g):,} รายการ ({time.time()-t0:.0f} s)", flush=True)
        return g
    return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")


def save_table(df, name, index=False):
    """บันทึกตารางลง OUTDIR (ถ้า SAVE_TO_DRIVE=True คือ Drive: MyDrive/PM25_Saraburi_notebooks)"""
    path = os.path.join(OUTDIR, name)
    if name.endswith(".gpkg"):
        df.to_file(path, driver="GPKG")
    else:
        df.to_csv(path, index=index, encoding="utf-8-sig")
    return path

## 1. ขอบเขตตำบล ประชากร และเส้นถนน

- **ขอบเขตตำบล** ชุดเดียวกับ Notebook 06 และ 13–20 (OCHA COD-AB → GADM)
- **ประชากร** WorldPop 2020 UN-adjusted 100 m (อ่านเฉพาะกรอบจังหวัด)
- **ถนน** OpenStreetMap จาก Geofabrik (ไฟล์ shapefile ทั้งประเทศ อัปเดตทุกวัน) เฉพาะ motorway/trunk/primary — สำรอง: ohsome → Overpass

In [ ]:
TAMBON, BND_SRC = load_tambons()
TAMBON = TAMBON.reset_index(drop=True); TAMBON["tidx"] = np.arange(len(TAMBON))
TAMBON["area_km2"] = TAMBON.to_crs(UTM).area / 1e6
w, s, e, n = TAMBON.total_bounds
PROV_BOUNDS = (w - 0.02, s - 0.02, e + 0.02, n + 0.02)
WP, WP_TR, WP_CRS, POP_SRC = worldpop_window(PROV_BOUNDS, prefer_100m=WORLDPOP_100M)
from rasterio import features
TID = features.rasterize(((g, i + 1) for i, g in zip(TAMBON.tidx, TAMBON.to_crs(WP_CRS).geometry)),
                         out_shape=WP.shape, transform=WP_TR, fill=0, dtype="int32") - 1

BBOX = f"({s - 0.03:.4f},{w - 0.03:.4f},{n + 0.03:.4f},{e + 0.03:.4f})"
ROADS = overpass_gdf([f'way["highway"~"^({"|".join(MAJOR_CLASSES)})$"]'], BBOX, "line",
                     ohsome_filter="highway in (" + ", ".join(MAJOR_CLASSES) + ")",
                     gf=("roads", MAJOR_CLASSES, "highway"))
if ROADS is None or ROADS.empty:
    raise RuntimeError("ดึงเส้นถนน OSM ไม่ได้ — รันเซลล์นี้ซ้ำภายหลัง")
for c in ["ref", "name", "highway"]:
    if c not in ROADS: ROADS[c] = None
ROADS = ROADS[ROADS.geometry.type.isin(["LineString", "MultiLineString"])].to_crs(UTM)
print(f"✓ ถนนสายหลัก {len(ROADS):,} เส้น ยาวรวม {ROADS.length.sum() / 1000:,.0f} km | {BND_SRC} | {POP_SRC}")

## 2. เลือกเส้นทางหลวงหมายเลข 1 และ 2

ใช้แท็ก `ref` ของ OSM (เลขทางหลวง เช่น `1`, `2`, หรือ `AH2;1`) เป็นหลัก และชื่อถนน (`พหลโยธิน`, `มิตรภาพ`) เป็นตัวช่วย
**ต้องดูแผนที่ทุกครั้ง** — OSM บางช่วงอาจไม่มี ref หรือทางขนาน (frontage road) ใช้ชื่อเดียวกัน

In [ ]:
def match_road(df, refs, names):
    ref_ok = df["ref"].fillna("").astype(str).str.split(r"[;,/]").apply(lambda xs: any(x.strip() in refs for x in xs))
    name_ok = df["name"].fillna("").astype(str).str.lower().apply(lambda s_: any(k.lower() in s_ for k in names))
    return ref_ok | name_ok

for k, spec in TARGET_ROADS.items():
    ROADS[k] = match_road(ROADS, spec["refs"], spec["names"])
ROADS["MAIN"] = ROADS[list(TARGET_ROADS)].any(axis=1)
prov = TAMBON.to_crs(UTM).union_all() if hasattr(TAMBON.geometry, "union_all") else TAMBON.to_crs(UTM).unary_union
for k in list(TARGET_ROADS) + ["MAIN"]:
    print(f"{k}: {ROADS[k].sum()} เส้น, ยาวในจังหวัด {ROADS[ROADS[k]].intersection(prov).length.sum() / 1000:.1f} km")

fig, ax = plt.subplots(figsize=(8, 7.5))
TAMBON.to_crs(UTM).boundary.plot(ax=ax, color="#bbbbbb", lw=.4)
ROADS[~ROADS.MAIN].plot(ax=ax, color="#999999", lw=.6, label="other major roads")
ROADS[ROADS.HW1].plot(ax=ax, color="#d7301f", lw=2, label="HW1 Phahon Yothin")
ROADS[ROADS.HW2].plot(ax=ax, color="#0570b0", lw=2, label="HW2 Mittraphap")
st = gpd.GeoSeries([Point(xy) for xy in STATIONS_XY.values()], crs="EPSG:4326").to_crs(UTM)
ax.scatter(st.x, st.y, marker="^", s=80, c="cyan", edgecolor="k", zorder=5)
for (k_, _), p in zip(STATIONS_XY.items(), st): ax.annotate(k_, (p.x, p.y), xytext=(4, 4), textcoords="offset points", fontweight="bold")
ax.legend(fontsize=8); ax.set_title("Check: selected highways 1 and 2 (OpenStreetMap)"); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 3. แผนที่ระยะห่างจากถนน (20 m)

แปลงเส้นถนนเป็นกริด 20 m ในพิกัด UTM 47N แล้วคำนวณระยะ Euclidean จากทุกเซลล์ถึงเซลล์ถนนที่ใกล้ที่สุด (distance transform)
ความคลาดเคลื่อนจากการแปลงเป็นกริด ≤ ~15 m ซึ่งเล็กกว่าแถบระยะที่ใช้ (≥ 50 m)

In [ ]:
from rasterio.transform import from_origin
pb = TAMBON.to_crs(UTM).total_bounds
X0, Y1 = np.floor(pb[0] / DIST_RES) * DIST_RES - 2000, np.ceil(pb[3] / DIST_RES) * DIST_RES + 2000
NX = int(np.ceil((pb[2] + 2000 - X0) / DIST_RES)); NY = int(np.ceil((Y1 - (pb[1] - 2000)) / DIST_RES))
DTR = from_origin(X0, Y1, DIST_RES, DIST_RES)
print(f"กริดระยะห่าง {NX:,} × {NY:,} = {NX * NY / 1e6:.1f} ล้านเซลล์")

def distance_raster(lines):
    if lines.empty: return np.full((NY, NX), np.inf, dtype="float32")
    m = features.rasterize(((g, 1) for g in lines.geometry), out_shape=(NY, NX), transform=DTR, fill=0, all_touched=True, dtype="uint8")
    return (ndimage.distance_transform_edt(m == 0) * DIST_RES).astype("float32")

DIST = {k: distance_raster(ROADS[ROADS[k]]) for k in list(TARGET_ROADS) + ["MAIN"]}
DIST["MAJOR"] = distance_raster(ROADS)
print("✓ แผนที่ระยะห่าง:", list(DIST))

## 4. จุดย่อยประชากร

เซลล์ WorldPop 100 m กว้างกว่าแถบ 0–115 m ถ้าตัดสินจาก **จุดศูนย์กลางเซลล์** อย่างเดียว คนทั้งเซลล์จะถูกนับว่า "ใน" หรือ "นอก" แนวกันชนทั้งหมด
จึงแบ่งแต่ละเซลล์เป็น 5 × 5 จุดย่อย (ห่างกัน ~18 m) ให้แต่ละจุดได้ประชากร 1/25 ของเซลล์ (สมมติว่ากระจายสม่ำเสมอภายในเซลล์) แล้ววัดระยะของทุกจุดย่อย

In [ ]:
ii, jj = np.nonzero((TID >= 0) & (WP > 0))
off = (np.arange(SUBPIX) + .5) / SUBPIX
oy, ox = np.meshgrid(off, off, indexing="ij"); oy, ox = oy.ravel(), ox.ravel()
lon = (WP_TR.c + (jj[:, None] + ox[None, :]) * WP_TR.a).ravel()
lat = (WP_TR.f + (ii[:, None] + oy[None, :]) * WP_TR.e).ravel()
PT_POP = np.repeat(WP[ii, jj] / SUBPIX ** 2, SUBPIX ** 2)
PT_T = np.repeat(TID[ii, jj], SUBPIX ** 2)
xs, ys = Transformer.from_crs(WP_CRS, UTM, always_xy=True).transform(lon, lat)
col = np.clip(((xs - X0) / DIST_RES).astype(int), 0, NX - 1); row = np.clip(((Y1 - ys) / DIST_RES).astype(int), 0, NY - 1)
PT_D = {k: v[row, col] for k, v in DIST.items()}
# เทียบ: ใช้เฉพาะจุดศูนย์กลางเซลล์
cx, cy = Transformer.from_crs(WP_CRS, UTM, always_xy=True).transform(WP_TR.c + (jj + .5) * WP_TR.a, WP_TR.f + (ii + .5) * WP_TR.e)
CC = {"pop": WP[ii, jj], "t": TID[ii, jj],
      "d": DIST["MAIN"][np.clip(((Y1 - cy) / DIST_RES).astype(int), 0, NY - 1), np.clip(((cx - X0) / DIST_RES).astype(int), 0, NX - 1)]}
print(f"เซลล์ประชากร {len(ii):,} → จุดย่อย {len(PT_POP):,} | ประชากรรวม {PT_POP.sum():,.0f} คน")

## 5. ตัวชี้วัดรายตำบล

$$\text{RoadProp}_i(b)=\frac{\sum_{p\in i}P_p\,\mathbb{1}[d_p\le b]}{\sum_{p\in i}P_p}\qquad
T_i(\lambda)=\frac{\sum_{p\in i}P_p\,e^{-d_p/\lambda}}{\sum_{p\in i}P_p}$$

| ตัวแปร | ความหมาย |
|---|---|
| `pop` | ประชากร WorldPop ในตำบล |
| `dist_mean_HW1/HW2/MAIN` | ระยะเฉลี่ยถ่วงประชากรถึง ทล.1 / ทล.2 / ทล.1 หรือ ทล.2 ที่ใกล้กว่า (m) |
| `dist_median_MAIN` | มัธยฐานระยะของประชากร (m) |
| **`share_500m_MAIN`** | **สัดส่วนประชากรภายใน 500 m จาก ทล.1/ทล.2 (ตัวชี้วัดหลัก)** |
| `pop_500m_MAIN` | จำนวนคนภายใน 500 m |
| `share_0_150m_MAIN`, `share_150_300m_MAIN`, `share_300_500m_MAIN` | แถบระยะภายใน 500 m (รวมกันได้เท่ากับ `share_500m_MAIN`) |
| `share_150m_HW1`, `share_300m_HW1`, `share_500m_HW1` | สัดส่วนประชากรภายใน 150/300/500 m จาก **ทล.1** |
| `share_150m_HW2`, `share_300m_HW2`, `share_500m_HW2` | สัดส่วนประชากรภายใน 150/300/500 m จาก **ทล.2** |
| `pop_{150,300,500}m_{HW1,HW2,MAIN}` | จำนวนคนในแนวกันชนแต่ละระยะ |
| `decay_λ200` ฯลฯ | ดัชนีการสัมผัสการจราจรแบบลดหลั่น (0–1) |
| `share_500m_MAJOR` | ภายใน 500 m จากถนนสายหลักทุกสาย (motorway/trunk/primary) |

> ค่า HW1 และ HW2 วัดแยกกัน คนที่อยู่ใกล้ทั้งสองสาย (บริเวณทางแยก ทล.1/ทล.2 อ.เมืองสระบุรี) จึงถูกนับในทั้งสองคอลัมน์ ส่วน `MAIN` วัดจากสายที่ใกล้กว่า ไม่นับซ้ำ ดังนั้น `share_HW1 + share_HW2 ≥ share_MAIN` เสมอ

In [ ]:
NT = len(TAMBON)
def by_t(weights): return np.bincount(PT_T, weights=weights, minlength=NT)
POP = by_t(PT_POP)

def wmedian_by_group(values, weights, groups, n):
    out = np.full(n, np.nan); order = np.lexsort((values, groups))
    v, w_, g = values[order], weights[order], groups[order]
    starts = np.searchsorted(g, np.arange(n)); ends = np.searchsorted(g, np.arange(n), side="right")
    for k in range(n):
        if ends[k] > starts[k]:
            cw = np.cumsum(w_[starts[k]:ends[k]]); out[k] = v[starts[k]:ends[k]][np.searchsorted(cw, cw[-1] / 2)]
    return out

R = TAMBON[["tidx", "name"] + (["a_code"] if "a_code" in TAMBON else []) + ["area_km2", "geometry"]].copy()
R["pop"] = POP
with np.errstate(invalid="ignore", divide="ignore"):
    for k in list(TARGET_ROADS) + ["MAIN", "MAJOR"]:
        R[f"dist_mean_{k}"] = by_t(PT_POP * PT_D[k]) / POP
    R["dist_median_MAIN"] = wmedian_by_group(PT_D["MAIN"], PT_POP, PT_T, NT)
    for b in BUFFERS:
        R[f"share_{b}m_MAIN"] = by_t(PT_POP * (PT_D["MAIN"] <= b)) / POP
    R[f"pop_{MAIN_BUFFER}m_MAIN"] = by_t(PT_POP * (PT_D["MAIN"] <= MAIN_BUFFER))
    for a, b in BANDS:
        R[f"share_{a}_{b}m_MAIN"] = by_t(PT_POP * ((PT_D["MAIN"] > a if a else PT_D["MAIN"] >= 0) & (PT_D["MAIN"] <= b))) / POP
    for lam in LAMBDAS:
        R[f"decay_l{lam}_MAIN"] = by_t(PT_POP * np.exp(-PT_D["MAIN"] / lam)) / POP
    R[f"share_{MAIN_BUFFER}m_MAJOR"] = by_t(PT_POP * (PT_D["MAJOR"] <= MAIN_BUFFER)) / POP
    # สัดส่วนประชากรริมถนนแยกรายสาย (หัวข้อ 7.3.2, 7.3.5)
    for k in list(TARGET_ROADS) + ["MAIN"]:
        for b in ROAD_BUFFERS:
            R[f"pop_{b}m_{k}"] = by_t(PT_POP * (PT_D[k] <= b))
            R[f"share_{b}m_{k}"] = R[f"pop_{b}m_{k}"] / POP
R = gpd.GeoDataFrame(R, geometry="geometry", crs=TAMBON.crs)
tot = PT_POP.sum(); near = (PT_POP * (PT_D["MAIN"] <= MAIN_BUFFER)).sum()
print(f"ประชากรจังหวัดที่อยู่ภายใน {MAIN_BUFFER} m จาก ทล.1/ทล.2: {near:,.0f} คน ({near / tot * 100:.1f}%) | "
      f"ตำบลที่มีประชากรในแนวนี้: {(R[f'share_{MAIN_BUFFER}m_MAIN'] > 0).sum()} จาก {NT}")
# ตรวจ: ผลรวมแถบระยะต้องเท่ากับสัดส่วนภายในขอบแถบสุดท้าย
_band_sum = sum(R[f"share_{a}_{b}m_MAIN"].fillna(0) for a, b in BANDS)
assert np.allclose(_band_sum, R[f"share_{BANDS[-1][1]}m_MAIN"].fillna(0)), "ผลรวมแถบระยะไม่เท่ากับสัดส่วนรวม"
for b in ROAD_BUFFERS:
    assert (R[f"share_{b}m_HW1"].fillna(0) + R[f"share_{b}m_HW2"].fillna(0) >= R[f"share_{b}m_MAIN"].fillna(0) - 1e-9).all()

ROAD_SUM = pd.DataFrame([{"road": k, "buffer_m": b, "pop_in_buffer": (PT_POP * (PT_D[k] <= b)).sum(),
                          "province_share_%": (PT_POP * (PT_D[k] <= b)).sum() / tot * 100,
                          "tambons_with_share>0": int((R[f"share_{b}m_{k}"] > 0).sum())}
                         for k in list(TARGET_ROADS) + ["MAIN"] for b in ROAD_BUFFERS])
print("\nประชากรริมถนนแยกรายสาย (MAIN = ทล.1 หรือ ทล.2 สายที่ใกล้กว่า ไม่นับซ้ำ)")
display(ROAD_SUM.round(1))

show_cols = (["name", "pop", f"pop_{MAIN_BUFFER}m_MAIN", f"share_{MAIN_BUFFER}m_MAIN"]
             + [f"share_{a}_{b}m_MAIN" for a, b in BANDS]
             + [f"share_{MAIN_BUFFER}m_{k}" for k in TARGET_ROADS]
             + ["dist_mean_MAIN", "dist_median_MAIN", "decay_l200_MAIN"])
display(R.sort_values(f"share_{MAIN_BUFFER}m_MAIN", ascending=False)[show_cols].head(15).round(3))

## 6. ความไวต่อความกว้างแนวกันชน

ถ้าสหสัมพันธ์ลำดับ (Spearman) ระหว่างแนวกันชนต่าง ๆ กับ 500 m สูง (> 0.9) การเลือก 500 m ไม่เปลี่ยนว่าตำบลใดเสี่ยงกว่ากัน
ถ้าต่ำ ต้องรายงานผลหลายระยะ (สอดคล้องกับข้อจำกัด MAUP ที่ระบุไว้ในสไลด์)

In [ ]:
base = R[f"share_{MAIN_BUFFER}m_MAIN"]
SENS = pd.DataFrame([{"buffer_m": b, "province_share_%": (PT_POP * (PT_D["MAIN"] <= b)).sum() / tot * 100,
                      "tambons_with_share>0": int((R[f"share_{b}m_MAIN"] > 0).sum()),
                      "spearman_vs_500m": stats.spearmanr(R[f"share_{b}m_MAIN"], base, nan_policy="omit")[0]} for b in BUFFERS])
SENS = pd.concat([SENS, pd.DataFrame([{"buffer_m": f"decay λ={lam}", "spearman_vs_500m": stats.spearmanr(R[f"decay_l{lam}_MAIN"], base, nan_policy="omit")[0]}
                                       for lam in LAMBDAS])], ignore_index=True)
display(SENS.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
b_ = SENS[SENS.buffer_m.apply(lambda v: isinstance(v, (int, np.integer)))]
axes[0].plot(b_.buffer_m, b_["province_share_%"], "-o"); axes[0].axvline(MAIN_BUFFER, ls="--", c="gray")
axes[0].set_xlabel("buffer width (m)"); axes[0].set_ylabel("% of province population"); axes[0].set_title("Population living near HW1/HW2")
axes[1].plot(b_.buffer_m, b_.spearman_vs_500m, "-o", c="k"); axes[1].axhline(.9, ls=":", c="crimson")
axes[1].set_xlabel("buffer width (m)"); axes[1].set_title("Rank agreement with the 500 m share (Spearman)")
plt.tight_layout(); plt.show()

## 7. ผลของความละเอียดกริด

เทียบสัดส่วนที่ได้จาก **จุดศูนย์กลางเซลล์ 100 m** กับ **จุดย่อย 5×5** — ถ้าต่างมากในแถบแคบที่สุด (0–150 m) แสดงว่าต้องใช้วิธีจุดย่อย (หรือข้อมูลอาคาร) ในการรายงาน

In [ ]:
NB = BANDS[0][1]   # แถบแคบที่สุด (0–150 m)
with np.errstate(invalid="ignore", divide="ignore"):
    POPc = np.bincount(CC["t"], weights=CC["pop"], minlength=NT)
    cmp = pd.DataFrame({
        "metric": [f"share ≤{MAIN_BUFFER} m", f"share 0–{NB} m"],
        "province_subpixel_%": [near / tot * 100, (PT_POP * (PT_D["MAIN"] <= NB)).sum() / tot * 100],
        "province_cell_centre_%": [(CC["pop"] * (CC["d"] <= MAIN_BUFFER)).sum() / CC["pop"].sum() * 100,
                                   (CC["pop"] * (CC["d"] <= NB)).sum() / CC["pop"].sum() * 100],
        "tambon_spearman": [stats.spearmanr(np.bincount(CC["t"], weights=CC["pop"] * (CC["d"] <= MAIN_BUFFER), minlength=NT) / POPc, base, nan_policy="omit")[0],
                            stats.spearmanr(np.bincount(CC["t"], weights=CC["pop"] * (CC["d"] <= NB), minlength=NT) / POPc,
                                            R[f"share_0_{NB}m_MAIN"], nan_policy="omit")[0]]})
display(cmp.round(3))

## 8. แผนที่ บันทึกไฟล์ และข้อความในบทวิธีวิจัย

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6.2))
R.plot(f"share_{MAIN_BUFFER}m_MAIN", ax=axes[0], cmap="OrRd", legend=True, edgecolor="k", lw=.2,
       legend_kwds={"shrink": .6, "label": "share of population"})
axes[0].set_title(f"Share of population ≤ {MAIN_BUFFER} m from HW1/HW2")
R.plot("dist_mean_MAIN", ax=axes[1], cmap="viridis_r", legend=True, edgecolor="k", lw=.2, legend_kwds={"shrink": .6, "label": "m"})
axes[1].set_title("Population-weighted mean distance to HW1/HW2")
R.plot("decay_l200_MAIN", ax=axes[2], cmap="magma_r", legend=True, edgecolor="k", lw=.2, legend_kwds={"shrink": .6, "label": "index 0–1"})
axes[2].set_title("Traffic exposure decay index ω(d), λ = 200 m")
rd = ROADS[ROADS.MAIN].to_crs(4326)
for ax in axes:
    rd.plot(ax=ax, color="k", lw=.8)
    for k_, (x, y) in STATIONS_XY.items(): ax.scatter(x, y, marker="^", s=60, c="cyan", edgecolor="k", zorder=5)
    ax.set_xlim(PROV_BOUNDS[0], PROV_BOUNDS[2]); ax.set_ylim(PROV_BOUNDS[1], PROV_BOUNDS[3]); ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
_p = ROAD_SUM.set_index(["road", "buffer_m"])["pop_in_buffer"]
top = R.sort_values(f"share_{MAIN_BUFFER}m_MAIN", ascending=False).head(5)
_r = SENS.loc[SENS.buffer_m.isin([300, 400, 600, 750]), "spearman_vs_500m"]; rho_rng = (_r.min(), _r.max())
REPORT = (f"ประชากรจังหวัดสระบุรี (WorldPop 2020) ที่อาศัยภายใน {MAIN_BUFFER} เมตรจากทางหลวงหมายเลข 1 หรือ 2 มีประมาณ {near:,.0f} คน "
          f"({near / tot * 100:.1f}% ของประชากรทั้งจังหวัด) กระจายใน {(R[f'share_{MAIN_BUFFER}m_MAIN'] > 0).sum()} จาก {NT} ตำบล "
          f"ตำบลที่สัดส่วนประชากรริมถนนสูงสุด ได้แก่ " + ", ".join(f"{r['name']} ({r[f'share_{MAIN_BUFFER}m_MAIN'] * 100:.0f}%)" for _, r in top.iterrows()) +
          (" การจัดลำดับตำบลตามสัดส่วนประชากรริมถนนสอดคล้องกันระหว่างแนวกันชน 300–750 m " if rho_rng[0] >= 0.9 else
           " การจัดลำดับตำบลเปลี่ยนไปตามความกว้างแนวกันชน (ควรรายงานหลายระยะ) ") +
          f"(Spearman {rho_rng[0]:.2f}–{rho_rng[1]:.2f} เทียบกับ 500 m) "
          f"เมื่อแยกรายสาย ประชากรภายใน {MAIN_BUFFER} m จากทางหลวงหมายเลข 1 มีประมาณ {_p[('HW1', MAIN_BUFFER)]:,.0f} คน "
          f"และจากทางหลวงหมายเลข 2 มีประมาณ {_p[('HW2', MAIN_BUFFER)]:,.0f} คน")
print(REPORT)
save_table(R, "roadside_tambon.gpkg"); save_table(R.drop(columns="geometry"), "roadside_tambon.csv")
save_table(SENS, "roadside_buffer_sensitivity.csv"); save_table(ROAD_SUM, "roadside_by_highway_summary.csv"); save_table(cmp, "roadside_grid_resolution_check.csv")
save_table(ROADS[ROADS.MAIN][["highway", "ref", "name", "HW1", "HW2", "geometry"]].to_crs(4326), "roadside_highways_used.gpkg")
json.dump({"boundaries": BND_SRC, "population": POP_SRC, "roads": "OpenStreetMap (Geofabrik thailand-latest-free / ohsome / Overpass)",
           "distance_resolution_m": DIST_RES, "subpixels_per_cell": SUBPIX ** 2},
          open(os.path.join(OUTDIR, "roadside_sources.json"), "w"), ensure_ascii=False, indent=1)
open(os.path.join(OUTDIR, "report_08_th.txt"), "w", encoding="utf-8").write(REPORT)
print("\nบันทึกแล้วใน", OUTDIR)

| ไฟล์ใน `OUTDIR` | ใช้ต่อใน |
|---|---|
| `roadside_tambon.gpkg` / `.csv` — ระยะ สัดส่วนริมถนน แถบระยะ ดัชนีลดหลั่น | Part 3 (ตัวแปรการสัมผัส/covariate: Notebook 17–19) |
| `roadside_buffer_sensitivity.csv`, `roadside_grid_resolution_check.csv` | ภาคผนวก (ความไว / MAUP) |
| `roadside_by_highway_summary.csv` — ประชากรริมถนนแยก ทล.1 / ทล.2 / รวม ที่ 150/300/500 m | บทที่ 4 (ตารางสรุป) |
| `roadside_highways_used.gpkg` | ตรวจเส้นถนนที่ใช้ใน QGIS |

**ตัวอย่างข้อความในบทวิธีวิจัย**

> ตัวชี้วัดการรับสัมผัสจากการจราจรคำนวณจากเส้นทางหลวงหมายเลข 1 (พหลโยธิน) และหมายเลข 2 (มิตรภาพ) ใน OpenStreetMap ร่วมกับข้อมูลประชากร
> WorldPop ปี 2020 ความละเอียด 100 เมตร โดยแบ่งแต่ละเซลล์ประชากรเป็น 25 จุดย่อยและคำนวณระยะจากแผนที่ระยะห่างความละเอียด 20 เมตร
> ตัวชี้วัดหลักคือสัดส่วนประชากรของตำบลที่อาศัยภายใน 500 เมตรจากทางหลวงหมายเลข 1 หรือ 2 ร่วมกับสัดส่วนในแถบระยะ 0–150, 150–300
> และ 300–500 เมตร และสัดส่วนที่คำนวณแยกรายสาย (ทางหลวงหมายเลข 1 และ 2) ที่แนวกันชน 150, 300 และ 500 เมตร
> พร้อมดัชนีการสัมผัสแบบลดหลั่นตามระยะ exp(−d/λ) ที่ λ = 100–500 เมตร และทดสอบความไวของผลต่อความกว้างแนวกันชน 50–1,000 เมตร

**ข้อจำกัด**
1. WorldPop กระจายประชากรด้วยแบบจำลอง (ไม่ใช่ทะเบียนบ้านรายหลัง) และเป็นปี 2020 ปีเดียว — ถ้าต้องการละเอียดขึ้น ใช้ข้อมูลอาคาร (เช่น Google Open Buildings) ถ่วงน้ำหนักในเซลล์
2. สมมติว่าประชากรกระจายสม่ำเสมอภายในเซลล์ 100 m ซึ่งไม่จริงในชุมชนที่บ้านเรือนเกาะแนวถนน (มักประเมินคนริมถนนต่ำไป)
3. ตัวชี้วัดเป็นระยะทางอย่างเดียว ยังไม่รวมปริมาณจราจรหรือสัดส่วนรถบรรทุก (ถ้าได้ข้อมูลกรมทางหลวง ให้ถ่วงด้วย AADT)
4. ความถูกต้องของเส้นถนนและแท็ก `ref` ใน OSM — ตรวจด้วยแผนที่ในขั้นที่ 2 และไฟล์ `roadside_highways_used.gpkg`

**เอกสารอ้างอิง**
- Karner, A. A., Eisinger, D. S., & Niemeier, D. A. (2010). Near-roadway air quality: Synthesizing the findings from real-world data. *Environmental Science & Technology, 44*(14), 5334–5344.
- HEI Panel on the Health Effects of Traffic-Related Air Pollution. (2010). *Traffic-related air pollution: A critical review of the literature on emissions, exposure, and health effects* (HEI Special Report 17). Health Effects Institute.
- Tatem, A. J. (2017). WorldPop, open data for spatial demography. *Scientific Data, 4*, 170004.
- OpenStreetMap contributors. (2026). *OpenStreetMap* [Data set]. Geofabrik GmbH extract for Thailand.